# 🎵 Music Streaming Behavior Analysis

## Análise do comportamento de usuários de streaming em Springfield e Shelbyville

Este projeto analisa dados de um serviço de streaming de música para identificar diferenças no volume de reproduções entre usuários de **Springfield** e **Shelbyville** ao longo de diferentes dias da semana.

O trabalho foi desenvolvido como parte do MBA em Ciência de Dados e foi reorganizado para apresentação em portfólio, preservando a lógica analítica e os resultados do projeto original.

### Objetivo

Investigar a seguinte hipótese:

> **A atividade dos usuários é diferente dependendo do dia da semana e da cidade.**

### Etapas da análise

1. Exploração inicial dos dados
2. Limpeza e preparação dos dados
3. Tratamento de valores ausentes e duplicados
4. Padronização de categorias
5. Comparação da atividade entre cidades e dias da semana
6. Interpretação dos resultados

> **Nota metodológica:** neste projeto, a hipótese é avaliada de forma descritiva a partir dos dados disponíveis. Não é aplicado um teste estatístico inferencial com nível de significância ou p-valor.


## 1. Importação e visão geral dos dados

Primeiro, importamos a biblioteca `pandas` e carregamos o conjunto de dados. Em seguida, avaliamos sua estrutura, quantidade de registros, tipos de dados e possíveis problemas de qualidade.


In [ ]:
import pandas as pd


In [ ]:
# O caminho abaixo considera a estrutura recomendada do repositório:
# music-listening-behavior-analysis/
# ├── data/music_project_en.csv
# └── notebook/music_streaming_analysis.ipynb

df = pd.read_csv("../data/music_project_en.csv")


In [ ]:
df.head(10)


In [ ]:
df.info()


### Observações iniciais

O conjunto de dados possui **65.079 registros** e 7 colunas. Cada linha representa uma reprodução de música e contém informações sobre usuário, faixa, artista, gênero, cidade, horário e dia da semana.

Na inspeção inicial foram identificados alguns pontos que precisam ser tratados antes da análise:

- nomes de colunas com capitalização inconsistente e espaços extras;
- valores ausentes nas colunas `Track`, `artist` e `genre`;
- possibilidade de registros duplicados;
- necessidade de verificar inconsistências na nomenclatura dos gêneros musicais.

Esses problemas justificam uma etapa de pré-processamento antes da comparação entre as cidades.


## 2. Preparação e limpeza dos dados

### 2.1 Padronização dos nomes das colunas

Os nomes das colunas são convertidos para letras minúsculas, têm espaços extras removidos e são ajustados para seguir o padrão `snake_case`.


In [ ]:
df.columns


In [ ]:
df.columns = [column.lower().strip() for column in df.columns]
df = df.rename(columns={"userid": "user_id"})

df.columns


### 2.2 Tratamento de valores ausentes

Primeiro, verificamos a quantidade de valores ausentes em cada coluna.


In [ ]:
df.isna().sum()


As colunas `track`, `artist` e `genre` possuem valores ausentes. Como o projeto não dispõe de uma fonte adicional para recuperar essas informações, os valores ausentes são preenchidos com `"unknown"`.

Essa decisão evita a remoção de registros completos, mas deve ser considerada na interpretação dos resultados, especialmente em análises que dependam diretamente do gênero musical.


In [ ]:
columns_to_replace = ["track", "artist", "genre"]

for column in columns_to_replace:
    df[column] = df[column].fillna("unknown")

df.isna().sum()


### 2.3 Remoção de registros duplicados

Registros duplicados podem distorcer contagens e gerar uma percepção incorreta do volume de atividade. Por isso, verificamos e removemos duplicatas explícitas.


In [ ]:
duplicated_before = df.duplicated().sum()
duplicated_before


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print(f"Registros após remoção de duplicados: {len(df)}")
print(f"Duplicados restantes: {df.duplicated().sum()}")


### 2.4 Padronização dos gêneros musicais

Além dos duplicados de linhas, o conjunto de dados apresenta categorias equivalentes escritas de formas diferentes. As variações `hip`, `hop` e `hip-hop` são padronizadas como `hiphop`.


In [ ]:
sorted(df["genre"].unique())[:30]


In [ ]:
def replace_wrong_genres(dataframe, wrong_genres, correct_genre):
    dataframe["genre"] = dataframe["genre"].replace(wrong_genres, correct_genre)

wrong_genres = ["hip", "hop", "hip-hop"]
replace_wrong_genres(df, wrong_genres, "hiphop")


In [ ]:
sorted(df["genre"].unique())[:30]


### Resultado do pré-processamento

Após a limpeza:

- os nomes das colunas foram padronizados;
- os valores ausentes foram tratados;
- os registros duplicados foram removidos;
- categorias inconsistentes de gênero foram unificadas.

Com isso, o conjunto de dados está mais adequado para a análise comparativa.


## 3. Análise do comportamento dos usuários

### 3.1 Atividade total por cidade

Primeiro, comparamos o número total de reproduções registradas em cada cidade.


In [ ]:
tracks_by_city = df.groupby("city")["track"].count().sort_values(ascending=False)
tracks_by_city


Springfield apresenta um volume total de reproduções consideravelmente maior que Shelbyville no conjunto de dados. Esse resultado, isoladamente, não permite concluir que uma cidade seja mais ativa em termos populacionais, pois não estamos controlando diferenças no número de usuários ou na representatividade da amostra.


### 3.2 Atividade total por dia da semana

Em seguida, analisamos o volume de reproduções nos três dias disponíveis: segunda-feira, quarta-feira e sexta-feira.


In [ ]:
tracks_by_day = df.groupby("day")["track"].count().sort_values(ascending=False)
tracks_by_day


No conjunto de dados, sexta-feira e segunda-feira apresentam volumes totais próximos, enquanto quarta-feira registra menor quantidade de reproduções quando as duas cidades são consideradas em conjunto.


### 3.3 Comparação entre cidade e dia

Para avaliar a hipótese do projeto, comparamos o número de reproduções em cada combinação de cidade e dia da semana.


In [ ]:
activity_by_city_day = (
    df.groupby(["day", "city"])["user_id"]
      .count()
      .unstack()
      .reindex(["Monday", "Wednesday", "Friday"])
)

activity_by_city_day


Os resultados mostram padrões diferentes entre as cidades:

- **Springfield:** 15.740 reproduções na segunda-feira, 11.056 na quarta-feira e 15.945 na sexta-feira.
- **Shelbyville:** 5.614 reproduções na segunda-feira, 7.003 na quarta-feira e 5.895 na sexta-feira.

Springfield apresenta seu maior volume na sexta-feira e também alta atividade na segunda-feira. Shelbyville, por outro lado, apresenta seu maior volume na quarta-feira.


## 4. Principais achados

A análise descritiva mostra que:

- Springfield possui maior volume absoluto de reproduções no conjunto de dados analisado;
- o padrão de atividade varia ao longo dos três dias disponíveis;
- Springfield apresenta maior atividade na segunda e na sexta-feira;
- Shelbyville apresenta seu maior volume na quarta-feira;
- portanto, os dados observados são **compatíveis com a hipótese de que a atividade varia conforme a cidade e o dia da semana**.

É importante evitar interpretar esse resultado como uma confirmação estatística sobre toda a população das cidades, pois o projeto utiliza dados de uma única fonte e não aplica um teste estatístico inferencial.


## 5. Conclusão

O projeto identificou diferenças no comportamento de reprodução de músicas entre Springfield e Shelbyville nos dias analisados.

Antes da análise, foi necessário realizar um processo de preparação dos dados, incluindo padronização de colunas, tratamento de valores ausentes, remoção de duplicados e correção de categorias inconsistentes.

Os resultados descritivos indicam padrões distintos entre as cidades: enquanto Springfield concentra maior atividade na segunda e na sexta-feira, Shelbyville apresenta maior atividade na quarta-feira.

Assim, **dentro do conjunto de dados analisado**, os resultados dão suporte descritivo à hipótese proposta. Entretanto, não são suficientes para generalizar o comportamento para toda a população das cidades.

### Aprendizados

Este projeto permitiu praticar conceitos fundamentais de análise de dados com Python, incluindo:

- exploração inicial de datasets;
- identificação e tratamento de problemas de qualidade dos dados;
- manipulação de dados com `pandas`;
- tratamento de valores ausentes e duplicados;
- padronização de categorias;
- agrupamento e filtragem de dados;
- comparação de grupos;
- interpretação cuidadosa de resultados e limitações analíticas.
